# 00 · Setup del Data Agent FraSoHome

Carga los CSV en tablas Delta y crea, configura y publica el **Fabric Data Agent** con el SDK.

**Requisitos:** ejecutar dentro de Fabric, con el Lakehouse `lh_frasohome` asociado como predeterminado
(el despliegue lo hace automáticamente). Datos en `Files/frasohome/`.


## 1. Instalar el SDK
> Tras instalar, si Fabric lo pide, **reinicia el kernel** y continúa desde la celda 2.


In [ ]:
%pip install -q fabric-data-agent-sdk


## 2. Cargar los CSV en tablas Delta (capa gold)


In [ ]:
BASE = "Files/frasohome"

tablas = {
    "FS_Clientes.csv":            "DimCustomer",
    "FS_Tiendas.csv":             "DimStore",
    "FS_Productos.csv":           "DimProduct",
    "FS_VentasResumenDiario.csv": "FactSalesDaily",
    "FS_StockDiario.csv":         "FactStockDaily",
    "FS_DevolucionesFact.csv":    "FactReturns",
    "FS_Alertas_Stockout.csv":    "AlertasStockout",
}

for archivo, tabla in tablas.items():
    df = (spark.read.option("header", True).option("inferSchema", True)
                    .csv(f"{BASE}/{archivo}"))
    df.write.format("delta").mode("overwrite").saveAsTable(tabla)
    print(f"{tabla}: {df.count()} filas")


## 3. Identificadores del entorno


In [ ]:
import notebookutils

ctx = notebookutils.runtime.context
workspace_id = ctx["currentWorkspaceId"]
lakehouse_id = ctx["defaultLakehouseId"]
print("workspace:", workspace_id)
print("lakehouse :", lakehouse_id)


## 4. Crear, configurar y publicar el Data Agent


In [ ]:
from fabric.dataagent.client import create_data_agent

AGENT_NAME = "FraSoHome BI Agent"

agent = create_data_agent(data_agent_name=AGENT_NAME, workspace_id=workspace_id)


In [ ]:
agent_instructions = r"""You are the FraSoHome retail BI assistant. Answer only from the gold retail tables in this lakehouse.

Definitions:
- Net sales = SUM(NetSalesEUR) from FactSalesDaily. Orders = SUM(Orders).
- Channel is 'Online' or 'Tienda' (store). Never mix channels unless the user asks for an omnichannel view.
- Return rate (amount) = SUM(FactReturns.RefundAmountEUR) / SUM(FactSalesDaily.NetSalesEUR) for the same period and channel.
- Stockout risk comes from AlertasStockout; Priority = 'Alta' is the highest.
- FactSalesDaily is a daily summary by store and channel and has NO product detail. If asked for sales by product, say that grain is not available.

Tables:
- DimCustomer(CustomerID, Tier, JoinDate, City, ConsentMarketing)
- DimStore(StoreID, StoreName, City, Region)
- DimProduct(ProductID, ProductName, CategoryName, Material, Color, ListPriceEUR, StandardCostEUR)
- FactSalesDaily(Date, Channel, StoreID, GrossSalesEUR, DiscountsEUR, ReturnsEUR, NetSalesEUR, Orders)
- FactStockDaily(Date, StoreID, ProductID, OnHandUnits, AvailableUnits)
- FactReturns(ReturnDate, Channel, StoreID, CustomerID, ProductID, Units, RefundAmountEUR, ReasonDesc, Approved)
- AlertasStockout(AlertDate, StoreID, ProductID, AvailableUnits, SalesUnitsLast7d, Priority)

Answer format: direct answer first, then evidence (period, metric, filter), then any missing-data caveat. Separate facts from hypotheses.

Example questions (natural language -> SQL):
1) Net online sales between 2026-02-01 and 2026-02-15 ->
   SELECT SUM(NetSalesEUR) FROM FactSalesDaily WHERE Channel='Online' AND Date BETWEEN '2026-02-01' AND '2026-02-15';
2) Which store had the highest net sales on 2026-02-10? ->
   SELECT TOP 1 StoreID, SUM(NetSalesEUR) AS NetSales FROM FactSalesDaily WHERE Date='2026-02-10' AND Channel='Tienda' GROUP BY StoreID ORDER BY NetSales DESC;
3) Which return reasons drive the highest refund amount? ->
   SELECT ReasonDesc, SUM(RefundAmountEUR) AS TotalRefund FROM FactReturns WHERE Approved='Si' GROUP BY ReasonDesc ORDER BY TotalRefund DESC;
4) Which products have a high-priority stockout alert? ->
   SELECT a.StoreID, a.ProductID, p.ProductName FROM AlertasStockout a JOIN DimProduct p ON a.ProductID=p.ProductID WHERE a.Priority='Alta';"""

# Instrucciones del agente (reglas de negocio + ejemplos NL->SQL)
agent.update_settings(ai_instructions=agent_instructions)

# Añadir el Lakehouse como origen de datos (staging)
agent.add_staging_datasource(
    artifact_name_or_id=lakehouse_id,
    workspace_id_or_name=workspace_id,
)

# Publicar
agent.publish_staging(description="Publicación inicial · FraSoHome BI Agent")
print("Agente publicado.")


## 5. Endpoint MCP del agente
El agente se consume por su endpoint MCP tras publicarlo.


In [ ]:
# Necesitamos el ID del artefacto del agente para construir el endpoint MCP.
import sempy.fabric as fabric
items = fabric.list_items(workspace=workspace_id)
agent_row = items[items["Display Name"] == AGENT_NAME]
if len(agent_row):
    agent_id = agent_row.iloc[0]["Id"]
    print("Data Agent ID:", agent_id)
    print(f"MCP: https://api.fabric.microsoft.com/v1/mcp/workspaces/{workspace_id}/dataagents/{agent_id}/agent")


## 6. Siguientes pasos
- Abre el **FraSoHome BI Agent** en el portal y prueba preguntas en lenguaje natural (en inglés).
- Ejecuta el notebook **`01_evaluate_agent`** para medir la calidad con el set de referencia.

Ejemplos de preguntas:
- *What were the net online sales between 2026-02-01 and 2026-02-15?*
- *Which store had the highest net sales on 2026-02-10?*
- *Which return reasons drive the highest refund amount?*
- *Which products have a high-priority stockout alert?*

Prueba negativa (límite de datos): *Show me the sales by product for MAD01* → el agente debe decir que
`FactSalesDaily` no tiene granularidad de producto.
